In [12]:
# Install PyTorch, TorchVision (for images), Matplotlib (plots), Scikit-learn (ML tools), and TQDM (progress bars)
pip install torch torchvision matplotlib scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Import the pandas library for handling CSVs and data manipulation
import pandas as pd

# Define the path to the training CSV file using a raw string to avoid escape character issues
TRAIN_CSV = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\train_labels.csv'

# Read the CSV file into a pandas DataFrame
train_df = pd.read_csv(TRAIN_CSV)

# Print the list of column names present in the DataFrame
print("Columns in train_labels.csv:", train_df.columns.tolist())

# Show first few rows to examine contents
print(train_df.head())


Columns in train_labels.csv: ['image_id', 'soil_type']
           image_id      soil_type
0  img_ed005410.jpg  Alluvial soil
1  img_0c5ecd2a.jpg  Alluvial soil
2  img_ed713bb5.jpg  Alluvial soil
3  img_12c58874.jpg  Alluvial soil
4  img_eff357af.jpg  Alluvial soil


In [10]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# CUDA setup (Set up GPU if available, else use CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Display selected device
print("Device:", device)

# Define file paths for training, testing, and submission
TRAIN_DIR = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\train'
TEST_DIR = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\test'
TRAIN_CSV = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\train_labels.csv'
TEST_CSV = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\test_ids.csv'
SUBMISSION_CSV = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\test_ids.csv'

# Set constants: image size, batch size, number of classes, epochs
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 30
LABELS = ['Alluvial soil', 'Black Soil', 'Clay soil', 'Red soil']

# 🔁 Create label-to-index and index-to-label dictionaries
label2idx = {label: i for i, label in enumerate(LABELS)}
idx2label = {i: label for label, i in label2idx.items()}

# 🧱 Define a custom Dataset class for loading soil images and labels
class SoilDataset(Dataset):
    def __init__(self, df, img_dir, labels=True, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.has_labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['image_id']
        img_path = os.path.join(self.img_dir, img_id)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.has_labels:
            label = label2idx[self.df.iloc[idx]['soil_type']]
            return image, label
        else:
            return image, img_id

# 🔧 Define data augmentations and normalization for training
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
# Define simpler transformation for validation and test
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# 📊 Load training CSV and split into training and validation sets
train_df = pd.read_csv(TRAIN_CSV)
train_data, val_data = train_test_split(train_df, test_size=0.2, stratify=train_df['soil_type'], random_state=42)

# Create Dataset and DataLoader objects for training and validation
train_dataset = SoilDataset(train_data, TRAIN_DIR, labels=True, transform=transform)
val_dataset = SoilDataset(val_data, TRAIN_DIR, labels=True, transform=val_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# 🧠 Define a simple custom CNN architecture
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)
# Load pretrained model and modify final layer for soil classification

def get_model(base='mobilenet'):
    if base == 'mobilenet':
        model = models.mobilenet_v2(pretrained=True)
        model.classifier[1] = nn.Linear(model.last_channel, NUM_CLASSES)
    elif base == 'resnet':
        model = models.resnet50(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

# 🏋️‍♂️ Define training function with early stopping
def train_model(model, name):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0
    patience = 5
    stop_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        correct, total = 0, 0
        for images, labels in tqdm(train_loader, desc=f"{name} Epoch {epoch+1}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        val_acc = evaluate(model)
        print(f"[{name}] Epoch {epoch+1}: Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f'{name}_best.pth')
            stop_counter = 0
        else:
            stop_counter += 1
            if stop_counter >= patience:
                print(f"Early stopping {name}")
                break

    print(f"Best Val Acc for {name}: {best_acc:.2f}%")
    return model

# 📈 Evaluate model on validation set
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

# 🔮 Predict on test set using ensemble of models and save submission
def predict_test(models):
    for model in models:
        model.eval()

    test_df = pd.read_csv(TEST_CSV)
    test_dataset = SoilDataset(test_df, TEST_DIR, labels=False, transform=val_transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    all_preds, all_ids = [], []

    with torch.no_grad():  # <--- Add this
        for images, img_ids in tqdm(test_loader, desc="Predicting Test Set"):
            images = images.to(device)

            # Run each model sequentially to avoid memory overflow
            outputs = []
            for m in models:
                m.to(device)
                out = m(images)
                outputs.append(torch.softmax(out, dim=1))
                torch.cuda.empty_cache()  # Clear unused memory

            avg_output = sum(outputs) / len(outputs)
            preds = torch.argmax(avg_output, dim=1).cpu().numpy()
            all_preds.extend([idx2label[i] for i in preds])
            all_ids.extend(img_ids)

    submission = pd.DataFrame({'image_id': all_ids, 'label': all_preds})
    submission.to_csv(SUBMISSION_CSV, index=False)
    print(f"Saved predictions to {SUBMISSION_CSV}")

# 🚀 Main script: train all models, load best checkpoints, and run prediction
if __name__ == '__main__':
    model1 = train_model(CustomCNN(), 'CustomCNN')
    model2 = train_model(get_model('mobilenet'), 'MobileNetV2')
    model3 = train_model(get_model('resnet'), 'ResNet50')

    # Load best models
    model1.load_state_dict(torch.load("CustomCNN_best.pth"))
    model2.load_state_dict(torch.load("MobileNetV2_best.pth"))
    model3.load_state_dict(torch.load("ResNet50_best.pth"))

    # Predict and savefepoch
    predict_test([model1.to(device), model2.to(device), model3.to(device)])


Device: cpu


CustomCNN Epoch 1: 100%|██████████| 31/31 [00:58<00:00,  1.90s/it]


[CustomCNN] Epoch 1: Train Acc: 18.94%, Val Acc: 18.78%


CustomCNN Epoch 2: 100%|██████████| 31/31 [00:48<00:00,  1.57s/it]


[CustomCNN] Epoch 2: Train Acc: 43.19%, Val Acc: 54.69%


...

[ResNet50] Epoch 12: Train Acc: 99.59%, Val Acc: 98.37%


ResNet50 Epoch 13: 100%|██████████| 31/31 [03:44<00:00,  7.25s/it]


[ResNet50] Epoch 13: Train Acc: 99.08%, Val Acc: 98.78%
Early stopping ResNet50
Best Val Acc for ResNet50: 98.78%


Predicting Test Set: 100%|██████████| 11/11 [00:40<00:00,  3.70s/it]

Saved predictions to C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\test_ids.csv


In [14]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

# ✅ Setup
# Set device to CPU explicitly
device = torch.device("cpu")
print("Device:", device)

# ✅ Paths
# Path to test images directory
TEST_DIR = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\test'
# Path to test CSV containing image IDs
TEST_CSV = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\test_ids.csv'
# Output submission path (CSV with predictions)
SUBMISSION_CSV = r'C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\test_ids.csv'

# Image size to resize inputs
IMG_SIZE = 224
# Batch size for prediction
BATCH_SIZE = 16  # Reduced for safety on GPU

# Label classes
LABELS = ['Alluvial soil', 'Black Soil', 'Clay soil', 'Red soil']
# Map labels to indices
label2idx = {label: i for i, label in enumerate(LABELS)}
# Map indices back to labels
idx2label = {i: label for label, i in label2idx.items()}

# ✅ Dataset Class
# Custom Dataset for test data
class SoilDataset(Dataset):
    def __init__(self, df, img_dir, labels=False, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.has_labels = labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Get image ID and path
        img_id = self.df.iloc[idx]['image_id']
        img_path = os.path.join(self.img_dir, img_id)
        # Open and convert to RGB
        image = Image.open(img_path).convert('RGB')
        # Apply transformations
        if self.transform:
            image = self.transform(image)
        # Return image and ID (no label in test)
        return image, img_id

# ✅ Transforms
# Resize and normalize images
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ✅ Models
# Custom CNN architecture
class CustomCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, len(LABELS))
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)

# Get MobileNet or ResNet with modified output layer
def get_model(base='mobilenet'):
    if base == 'mobilenet':
        model = models.mobilenet_v2(pretrained=False)
        model.classifier[1] = nn.Linear(model.last_channel, len(LABELS))
    elif base == 'resnet':
        model = models.resnet50(pretrained=False)
        model.fc = nn.Linear(model.fc.in_features, len(LABELS))
    return model

# ✅ Predict Function
# Function to run predictions on test set
def predict_test(models):
    # Set models to eval mode and CPU
    for model in models:
        model.eval()
        model.to(device)

    # Load test CSV and create dataset/loader
    test_df = pd.read_csv(TEST_CSV)
    test_dataset = SoilDataset(test_df, TEST_DIR, labels=False, transform=val_transform)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    all_preds, all_ids = [], []

    with torch.no_grad():
        for images, img_ids in tqdm(test_loader, desc="Predicting Test Set"):
            images = images.to(device)
            outputs = []
            for m in models:
                out = m(images)
                outputs.append(torch.softmax(out, dim=1))
            avg_output = sum(outputs) / len(outputs)
            preds = torch.argmax(avg_output, dim=1).numpy()
            all_preds.extend([idx2label[i] for i in preds])
            all_ids.extend(img_ids)

    # Save predictions to CSV
    submission = pd.DataFrame({'image_id': all_ids, 'label': all_preds})
    submission.to_csv(SUBMISSION_CSV, index=False)
    print(f"✅ Saved predictions to {SUBMISSION_CSV}")

# ✅ Run Prediction
if __name__ == '__main__':
    # Load CustomCNN and its weights
    model1 = CustomCNN()
    model1.load_state_dict(torch.load("CustomCNN_best.pth", map_location=device))

    # Load MobileNetV2 model and weights
    model2 = get_model('mobilenet')
    model2.load_state_dict(torch.load("MobileNetV2_best.pth", map_location=device))

    # Load ResNet50 model and weights
    model3 = get_model('resnet')
    model3.load_state_dict(torch.load("ResNet50_best.pth", map_location=device))

    # Predict on test set with ensemble of models
    predict_test([model1, model2, model3])


Device: cpu


C:\Users\akans\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\akans\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Predicting Test Set: 100%|██████████| 22/22 [00:51<00:00,  2.33s/it]

✅ Saved predictions to C:\Users\akans\OneDrive\Desktop\WorkSpace\soil_detection\soil_classification-2025\test_ids.csv
